# Q3D + JosephsonCircuits Plotly Comparison

This notebook compares Layout admittance extraction with Q3D capacitance + JosephsonCircuits simulation for the PF6FQ thesis analysis.

In [ ]:
from __future__ import annotations

import importlib
import inspect
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists() and (path / "core").exists()
)
STUDY_DIR = REPO_ROOT / "sandbox/thesis_pf6fq_external_coupling_analysis"
for import_path in (REPO_ROOT, STUDY_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

import config as thesis_plot_config  # noqa: E402
import thesis_comparison_analysis as comparison_analysis  # noqa: E402

thesis_plot_config = importlib.reload(thesis_plot_config)
comparison_analysis = importlib.reload(comparison_analysis)

from config import plotly_show_config  # noqa: E402
from q3d_xy_external_coupling import (  # noqa: E402
    DEFAULT_L_JUN_NH_VALUES,
    DEFAULT_QUBITS,
    DEFAULT_SYNTHETIC_C_EFF_XY_FF_VALUES,
    SweepProgressEvent,
    load_floating_xy_capacitances,
    run_q3d_xy_simulation_sweep,
    run_synthetic_c_eff_xy_simulation_sweep,
)
from thesis_comparison_analysis import (  # noqa: E402
    add_t1_from_ceff_references,
    build_c_eff_reference_table,
    build_c_eff_xy_t1_fit_parameter_summary,
    build_c_eff_xy_t1_result_display_tables,
    build_c_eff_xy_t1_trend_table,
    build_lc_frequency_fit_display_table,
    build_on_resonance_re_y_table,
    build_re_y_ratio_display_table,
    build_resonance_dataset,
    build_resonance_frequency_comparison,
    build_resonance_frequency_ratio_display_table,
    fit_c_eff_xy_t1_trend,
    fit_lc_frequency_sweeps,
    fit_t1_capacitance,
    load_circuit_observables,
    load_circuit_reduced_traces,
    load_layout_xy_im_y_traces,
    load_layout_xy_re_y_points,
    load_layout_xy_resonances,
    make_c_eff_overview_figure,
    make_c_eff_reference_diagnostic_figure,
    make_c_eff_xy_t1_trend_comparison_figure,
    make_c_eff_xy_t1_trend_figure,
    make_focus_admittance_trace_figure,
    make_focus_resonance_extraction_figure,
    make_im_trace_comparison_figure,
    make_on_resonance_re_y_figure,
    make_re_y_ratio_figure,
    make_resonance_frequency_ratio_figure,
    make_resonance_frequency_sweep_figure,
    make_t1_comparison_figure,
    match_layout_re_y_to_resonances,
    write_plotly_artifacts,
)

RAW_LAYOUT_DIR = REPO_ROOT / "data/raw/layout_simulation/PF6FQ"
RAW_OUTPUT_DIR = STUDY_DIR / "outputs/raw"
TABLE_OUTPUT_DIR = STUDY_DIR / "outputs/tables"
FIG_DIR = STUDY_DIR / "outputs/figures"
for output_dir in (RAW_OUTPUT_DIR, TABLE_OUTPUT_DIR, FIG_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)

RUN_SIMULATION = False
RUN_SYNTHETIC_C_EFF_XY_SIMULATION = False
SMOKE = False

QUBITS = ["Q0"] if SMOKE else list(DEFAULT_QUBITS)
CAPACITANCE_QUBITS = list(DEFAULT_QUBITS)
L_JUN_NH_VALUES = [24.0] if SMOKE else list(DEFAULT_L_JUN_NH_VALUES)

FOCUS_QUBIT = "Q0"
FOCUS_LJUN_NH = 24.0
TRACE_L_JUN_NH_VALUES = [FOCUS_LJUN_NH]
C_EFF_XY_T1_L_JUN_NH_VALUES = list(L_JUN_NH_VALUES)
C_EFF_XY_T1_SOURCES = ("Circuit", "Layout")
C_EFF_XY_T1_DISPLAY_L_JUN_NH = FOCUS_LJUN_NH
SYNTHETIC_TEMPLATE_QUBIT = "Q0"
SYNTHETIC_C_EFF_XY_FF_VALUES = list(DEFAULT_SYNTHETIC_C_EFF_XY_FF_VALUES)
SYNTHETIC_L_JUN_NH_VALUES = list(L_JUN_NH_VALUES)
SYNTHETIC_DISPLAY_L_JUN_NH = FOCUS_LJUN_NH
PLOT_TRACE_MAX_POINTS_PER_LINE = 3000

SWEEP_START_GHZ = 3.0 if SMOKE else 1.0
SWEEP_STOP_GHZ = 5.5 if SMOKE else 10.0
SWEEP_STEP_GHZ = 0.01 if SMOKE else 0.02
SYNTHETIC_SWEEP_START_GHZ = SWEEP_START_GHZ
SYNTHETIC_SWEEP_STOP_GHZ = SWEEP_STOP_GHZ
SYNTHETIC_SWEEP_STEP_GHZ = 0.01 if SMOKE else 0.05
PUMP_FREQ_GHZ = 8.001
SOURCE_CURRENT_AMP = 0.0
N_MODULATION_HARMONICS = 10
N_PUMP_HARMONICS = 20

N_PARALLEL_JUNCTIONS = 2
L_JUN_EFFECTIVE_FACTOR = thesis_plot_config.DEFAULT_L_JUN_EFFECTIVE_FACTOR
INCLUDE_FALLBACK_RESONANCES = False
RE_Y_MATCH_MAX_DELTA_MHZ = 30.0
WRITE_INTERACTIVE_HTML = True

print(f"Repository root: {REPO_ROOT}")
print(f"Study directory: {STUDY_DIR}")
print(f"Run simulation: {RUN_SIMULATION}, smoke: {SMOKE}")
print(f"Run synthetic Ceff,xy simulation: {RUN_SYNTHETIC_C_EFF_XY_SIMULATION}")
print(f"Focus point: {FOCUS_QUBIT}, L_jun = {FOCUS_LJUN_NH:.1f} nH")
print("LC fit policy: Circuit fixes Ls=0; Layout fits an effective Ls offset.")
print(
    f"Both routes use L_eff qubit term = {L_JUN_EFFECTIVE_FACTOR:.3f} * L_jun before any Layout offset."
)
print("L_jun is treated as per-junction inductance; default factor=0.5 for two parallel junctions.")
print(f"Comparison helper: {comparison_analysis.__file__}")
print(
    "Ceff,xy/T1 helper signature: "
    f"{inspect.signature(comparison_analysis.make_c_eff_xy_t1_trend_figure)}"
)


Repository root: /Users/arfiligol/Github/superconducting-circuits-tutorial
Study directory: /Users/arfiligol/Github/superconducting-circuits-tutorial/sandbox/thesis_pf6fq_external_coupling_analysis
Run simulation: False, smoke: False
Run synthetic Ceff,xy simulation: False
Focus point: Q0, L_jun = 24.0 nH
LC fit policy: Circuit fixes Ls=0; Layout fits an effective Ls offset.
Both routes use L_eff qubit term = 0.500 * L_jun before any Layout offset.
L_jun is treated as per-junction inductance; default factor=0.5 for two parallel junctions.
Comparison helper: /Users/arfiligol/Github/superconducting-circuits-tutorial/sandbox/thesis_pf6fq_external_coupling_analysis/thesis_comparison_analysis.py
Ceff,xy/T1 helper signature: (*, trend_df: 'pd.DataFrame', trend_fit_df: 'pd.DataFrame', no_offset_fit_df: 'pd.DataFrame | None' = None, l_jun_nh: 'float', source: 'str' = 'Circuit', title: 'str | None' = None, show_point_labels: 'bool' = True) -> 'go.Figure'


## Run Or Load Outputs

In [2]:
def write_csv(df: pd.DataFrame, path: Path) -> None:
    df.to_csv(path, index=False)
    print(f"Wrote {path.relative_to(REPO_ROOT)}")


def l_jun_figure_stem(l_jun_nh: float) -> str:
    return f"L{float(l_jun_nh):g}nH".replace(".", "p")


def make_notebook_progress_callback():
    last_stage = {"text": ""}

    def on_progress(event: SweepProgressEvent) -> None:
        text = f"[{event.case_index}/{event.case_total}] {event.stage}: {event.message}"
        if text != last_stage["text"]:
            print(text)
            last_stage["text"] = text

    return on_progress


cap_path = RAW_OUTPUT_DIR / "q3d_capacitance_parameters.csv"
jc_path = RAW_OUTPUT_DIR / "q3d_jc_xy_reduced_observables.csv"
trace_path = RAW_OUTPUT_DIR / "q3d_jc_xy_reduced_y_traces.csv"
selected_path = RAW_OUTPUT_DIR / "selected_qubit_resonances.csv"

if RUN_SIMULATION:
    sweep = run_q3d_xy_simulation_sweep(
        raw_layout_dir=RAW_LAYOUT_DIR,
        qubits=QUBITS,
        l_jun_nh_values=L_JUN_NH_VALUES,
        sweep_start_ghz=SWEEP_START_GHZ,
        sweep_stop_ghz=SWEEP_STOP_GHZ,
        sweep_step_ghz=SWEEP_STEP_GHZ,
        capacitance_summary_qubits=CAPACITANCE_QUBITS,
        repo_root=REPO_ROOT,
        pump_freq_ghz=PUMP_FREQ_GHZ,
        source_current_amp=SOURCE_CURRENT_AMP,
        n_modulation_harmonics=N_MODULATION_HARMONICS,
        n_pump_harmonics=N_PUMP_HARMONICS,
        progress=make_notebook_progress_callback(),
    )
    cap_df = pd.DataFrame(sweep.capacitance_rows)
    circuit_observables_source_df = pd.DataFrame(sweep.observable_rows)
    circuit_traces_source_df = pd.DataFrame(sweep.trace_rows)
    write_csv(cap_df, cap_path)
    write_csv(cap_df, TABLE_OUTPUT_DIR / "thesis_q3d_capacitance_summary.csv")
    write_csv(circuit_observables_source_df, jc_path)
    write_csv(circuit_traces_source_df, trace_path)
else:
    cap_df = pd.read_csv(cap_path)

layout_trace_df = load_layout_xy_im_y_traces(RAW_LAYOUT_DIR, qubits=CAPACITANCE_QUBITS)
layout_resonance_df = load_layout_xy_resonances(selected_path, qubits=CAPACITANCE_QUBITS)
layout_re_y_points_df = load_layout_xy_re_y_points(RAW_LAYOUT_DIR, qubits=CAPACITANCE_QUBITS)
layout_re_y_df = match_layout_re_y_to_resonances(
    layout_resonances_df=layout_resonance_df,
    layout_re_y_points_df=layout_re_y_points_df,
    max_delta_mhz=RE_Y_MATCH_MAX_DELTA_MHZ,
)
circuit_trace_df = load_circuit_reduced_traces(trace_path)
circuit_observables_df = load_circuit_observables(
    jc_path,
    include_fallback_resonances=INCLUDE_FALLBACK_RESONANCES,
)

resonance_df = build_resonance_dataset(
    layout_resonances_df=layout_resonance_df,
    circuit_observables_df=circuit_observables_df,
)
frequency_comparison_df = build_resonance_frequency_comparison(resonance_df)
frequency_ratio_display_df = build_resonance_frequency_ratio_display_table(resonance_df)
on_resonance_re_y_df = build_on_resonance_re_y_table(
    layout_re_y_df=layout_re_y_df,
    circuit_observables_df=circuit_observables_df,
)
re_y_ratio_display_df = build_re_y_ratio_display_table(on_resonance_re_y_df)
lc_fit_params_df, lc_fit_curve_df = fit_lc_frequency_sweeps(
    resonance_df,
    l_jun_effective_factor=L_JUN_EFFECTIVE_FACTOR,
)
lc_frequency_fit_display_df = build_lc_frequency_fit_display_table(
    lc_fit_params_df,
    dataset="Layout/Circuit",
    l_jun_effective_factor=L_JUN_EFFECTIVE_FACTOR,
)
c_eff_reference_df = build_c_eff_reference_table(
    lc_fit_params_df=lc_fit_params_df,
    circuit_observables_df=circuit_observables_df,
)
t1_df = add_t1_from_ceff_references(
    on_resonance_re_y_df=on_resonance_re_y_df,
    lc_fit_params_df=lc_fit_params_df,
)
t1_fit_df = fit_t1_capacitance(t1_df, lc_fit_params_df=lc_fit_params_df)
c_eff_xy_t1_trend_df = build_c_eff_xy_t1_trend_table(
    t1_df=t1_df,
    capacitance_df=cap_df,
    sources=C_EFF_XY_T1_SOURCES,
)
c_eff_xy_t1_trend_fit_df = fit_c_eff_xy_t1_trend(c_eff_xy_t1_trend_df)
c_eff_xy_t1_trend_fit_no_offset_df = fit_c_eff_xy_t1_trend(
    c_eff_xy_t1_trend_df,
    intercept_policy="zero",
)

write_csv(
    frequency_comparison_df,
    TABLE_OUTPUT_DIR / "thesis_layout_vs_circuit_frequency_comparison.csv",
)
write_csv(on_resonance_re_y_df, TABLE_OUTPUT_DIR / "thesis_on_resonance_re_y_comparison.csv")
write_csv(lc_fit_params_df, TABLE_OUTPUT_DIR / "thesis_lc_effective_capacitance_fit.csv")
write_csv(
    lc_frequency_fit_display_df,
    TABLE_OUTPUT_DIR / "thesis_lc_frequency_fit_display.csv",
)
write_csv(c_eff_reference_df, TABLE_OUTPUT_DIR / "thesis_c_eff_reference_comparison.csv")
write_csv(t1_df, TABLE_OUTPUT_DIR / "thesis_t1_comparison.csv")
write_csv(t1_fit_df, TABLE_OUTPUT_DIR / "thesis_t1_capacitance_fit.csv")
write_csv(c_eff_xy_t1_trend_df, TABLE_OUTPUT_DIR / "thesis_c_eff_xy_t1_trend.csv")
write_csv(c_eff_xy_t1_trend_fit_df, TABLE_OUTPUT_DIR / "thesis_c_eff_xy_t1_trend_fit.csv")
write_csv(
    c_eff_xy_t1_trend_fit_no_offset_df,
    TABLE_OUTPUT_DIR / "thesis_c_eff_xy_t1_trend_fit_no_offset.csv",
)
write_csv(
    frequency_comparison_df,
    TABLE_OUTPUT_DIR / "thesis_q3d_vs_hfss_frequency_comparison.csv",
)

print(f"Layout trace rows: {len(layout_trace_df):,}")
print(f"Circuit trace rows: {len(circuit_trace_df):,}")
print(f"Resonance rows: {len(resonance_df):,}")
print(f"On-resonance Re(Y) rows: {len(on_resonance_re_y_df):,}")

synthetic_cap_path = RAW_OUTPUT_DIR / "synthetic_c_eff_xy_capacitance_parameters.csv"
synthetic_jc_path = RAW_OUTPUT_DIR / "synthetic_c_eff_xy_reduced_observables.csv"
synthetic_trace_path = RAW_OUTPUT_DIR / "synthetic_c_eff_xy_reduced_y_traces.csv"
synthetic_outputs_exist = all(
    path.exists() for path in (synthetic_cap_path, synthetic_jc_path, synthetic_trace_path)
)
if RUN_SYNTHETIC_C_EFF_XY_SIMULATION or not synthetic_outputs_exist:
    synthetic_template = load_floating_xy_capacitances(
        RAW_LAYOUT_DIR,
        SYNTHETIC_TEMPLATE_QUBIT,
    )
    synthetic_sweep = run_synthetic_c_eff_xy_simulation_sweep(
        template=synthetic_template,
        target_c_eff_xy_ff_values=SYNTHETIC_C_EFF_XY_FF_VALUES,
        l_jun_nh_values=SYNTHETIC_L_JUN_NH_VALUES,
        sweep_start_ghz=SYNTHETIC_SWEEP_START_GHZ,
        sweep_stop_ghz=SYNTHETIC_SWEEP_STOP_GHZ,
        sweep_step_ghz=SYNTHETIC_SWEEP_STEP_GHZ,
        pump_freq_ghz=PUMP_FREQ_GHZ,
        source_current_amp=SOURCE_CURRENT_AMP,
        n_modulation_harmonics=N_MODULATION_HARMONICS,
        n_pump_harmonics=N_PUMP_HARMONICS,
        progress=make_notebook_progress_callback(),
    )
    synthetic_cap_df = pd.DataFrame(synthetic_sweep.capacitance_rows)
    synthetic_observables_source_df = pd.DataFrame(synthetic_sweep.observable_rows)
    synthetic_traces_source_df = pd.DataFrame(synthetic_sweep.trace_rows)
    write_csv(synthetic_cap_df, synthetic_cap_path)
    write_csv(synthetic_observables_source_df, synthetic_jc_path)
    write_csv(synthetic_traces_source_df, synthetic_trace_path)
else:
    synthetic_cap_df = pd.read_csv(synthetic_cap_path)

synthetic_observables_df = load_circuit_observables(
    synthetic_jc_path,
    include_fallback_resonances=INCLUDE_FALLBACK_RESONANCES,
)
synthetic_trace_df = load_circuit_reduced_traces(synthetic_trace_path)
synthetic_resonance_df = synthetic_observables_df[
    ["source", "qubit", "l_jun_nh", "frequency_ghz", "fallback", "crossed"]
].copy()
synthetic_lc_fit_params_df, synthetic_lc_fit_curve_df = fit_lc_frequency_sweeps(
    synthetic_resonance_df,
    l_jun_effective_factor=L_JUN_EFFECTIVE_FACTOR,
)
synthetic_lc_frequency_fit_display_df = build_lc_frequency_fit_display_table(
    synthetic_lc_fit_params_df,
    dataset="Synthetic Circuit",
    l_jun_effective_factor=L_JUN_EFFECTIVE_FACTOR,
)
synthetic_on_resonance_re_y_df = build_on_resonance_re_y_table(
    layout_re_y_df=pd.DataFrame(),
    circuit_observables_df=synthetic_observables_df,
)
synthetic_t1_df = add_t1_from_ceff_references(
    on_resonance_re_y_df=synthetic_on_resonance_re_y_df,
    lc_fit_params_df=synthetic_lc_fit_params_df,
)
synthetic_c_eff_xy_t1_trend_df = build_c_eff_xy_t1_trend_table(
    t1_df=synthetic_t1_df,
    capacitance_df=synthetic_cap_df,
)
synthetic_c_eff_xy_t1_fit_df = fit_c_eff_xy_t1_trend(synthetic_c_eff_xy_t1_trend_df)
synthetic_c_eff_xy_t1_fit_no_offset_df = fit_c_eff_xy_t1_trend(
    synthetic_c_eff_xy_t1_trend_df,
    intercept_policy="zero",
)
circuit_c_eff_xy_t1_fit_ab_df = build_c_eff_xy_t1_fit_parameter_summary(
    c_eff_xy_t1_trend_fit_df,
    dataset="Q3D Circuit",
)
circuit_c_eff_xy_t1_fit_no_offset_ab_df = build_c_eff_xy_t1_fit_parameter_summary(
    c_eff_xy_t1_trend_fit_no_offset_df,
    dataset="Q3D Circuit (B=0)",
)
synthetic_circuit_c_eff_xy_t1_fit_ab_df = build_c_eff_xy_t1_fit_parameter_summary(
    synthetic_c_eff_xy_t1_fit_df,
    dataset="Synthetic Circuit",
)
synthetic_circuit_c_eff_xy_t1_fit_no_offset_ab_df = build_c_eff_xy_t1_fit_parameter_summary(
    synthetic_c_eff_xy_t1_fit_no_offset_df,
    dataset="Synthetic Circuit (B=0)",
)

write_csv(synthetic_cap_df, TABLE_OUTPUT_DIR / "thesis_synthetic_c_eff_xy_capacitance.csv")
write_csv(
    synthetic_lc_fit_params_df,
    TABLE_OUTPUT_DIR / "thesis_synthetic_c_eff_xy_lc_fit.csv",
)
write_csv(
    synthetic_lc_frequency_fit_display_df,
    TABLE_OUTPUT_DIR / "thesis_synthetic_lc_frequency_fit_display.csv",
)
write_csv(synthetic_t1_df, TABLE_OUTPUT_DIR / "thesis_synthetic_c_eff_xy_t1.csv")
write_csv(
    synthetic_c_eff_xy_t1_trend_df,
    TABLE_OUTPUT_DIR / "thesis_synthetic_c_eff_xy_t1_trend.csv",
)
write_csv(
    synthetic_c_eff_xy_t1_fit_df,
    TABLE_OUTPUT_DIR / "thesis_synthetic_c_eff_xy_t1_fit.csv",
)
write_csv(
    synthetic_c_eff_xy_t1_fit_no_offset_df,
    TABLE_OUTPUT_DIR / "thesis_synthetic_c_eff_xy_t1_fit_no_offset.csv",
)
write_csv(
    circuit_c_eff_xy_t1_fit_ab_df,
    TABLE_OUTPUT_DIR / "thesis_circuit_c_eff_xy_t1_fit_ab.csv",
)
write_csv(
    circuit_c_eff_xy_t1_fit_no_offset_ab_df,
    TABLE_OUTPUT_DIR / "thesis_circuit_c_eff_xy_t1_fit_no_offset_ab.csv",
)
write_csv(
    synthetic_circuit_c_eff_xy_t1_fit_ab_df,
    TABLE_OUTPUT_DIR / "thesis_synthetic_circuit_c_eff_xy_t1_fit_ab.csv",
)
write_csv(
    synthetic_circuit_c_eff_xy_t1_fit_no_offset_ab_df,
    TABLE_OUTPUT_DIR / "thesis_synthetic_circuit_c_eff_xy_t1_fit_no_offset_ab.csv",
)

print(f"Synthetic capacitance rows: {len(synthetic_cap_df):,}")
print(f"Synthetic resonance rows: {len(synthetic_resonance_df):,}")
print(f"Synthetic trace rows: {len(synthetic_trace_df):,}")


Wrote sandbox/thesis_pf6fq_external_coupling_analysis/outputs/tables/thesis_layout_vs_circuit_frequency_comparison.csv
Wrote sandbox/thesis_pf6fq_external_coupling_analysis/outputs/tables/thesis_on_resonance_re_y_comparison.csv
Wrote sandbox/thesis_pf6fq_external_coupling_analysis/outputs/tables/thesis_lc_effective_capacitance_fit.csv
Wrote sandbox/thesis_pf6fq_external_coupling_analysis/outputs/tables/thesis_lc_frequency_fit_display.csv
Wrote sandbox/thesis_pf6fq_external_coupling_analysis/outputs/tables/thesis_c_eff_reference_comparison.csv
Wrote sandbox/thesis_pf6fq_external_coupling_analysis/outputs/tables/thesis_t1_comparison.csv
Wrote sandbox/thesis_pf6fq_external_coupling_analysis/outputs/tables/thesis_t1_capacitance_fit.csv
Wrote sandbox/thesis_pf6fq_external_coupling_analysis/outputs/tables/thesis_c_eff_xy_t1_trend.csv
Wrote sandbox/thesis_pf6fq_external_coupling_analysis/outputs/tables/thesis_c_eff_xy_t1_trend_fit.csv
Wrote sandbox/thesis_pf6fq_external_coupling_analysis/outp

## Result registry

This cell constructs every figure once, registers a compact display table payload for each figure, and defines `show_result(result_key)`. The result cells below only call that helper.


In [3]:
def table_payload(
    title: str,
    frame: pd.DataFrame,
    *,
    columns: list[str] | None = None,
    max_rows: int | None = None,
) -> dict[str, object]:
    data = frame.copy() if isinstance(frame, pd.DataFrame) else pd.DataFrame()
    if columns is not None:
        available_columns = [column for column in columns if column in data.columns]
        data = data.loc[:, available_columns]
    if max_rows is not None and len(data) > max_rows:
        data = data.head(max_rows).copy()
    return {"title": title, "data": data}


def filter_rows(
    frame: pd.DataFrame,
    *,
    source: str | None = None,
    qubit: str | None = None,
    l_jun_nh: float | None = None,
) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    mask = pd.Series(True, index=frame.index)
    if source is not None and "source" in frame.columns:
        mask &= frame["source"].astype(str) == str(source)
    if qubit is not None and "qubit" in frame.columns:
        mask &= frame["qubit"].astype(str) == str(qubit)
    if l_jun_nh is not None and "l_jun_nh" in frame.columns:
        mask &= (pd.to_numeric(frame["l_jun_nh"], errors="coerce") - float(l_jun_nh)).abs() < 1e-9
    return frame.loc[mask].copy()


def trace_summary(
    frame: pd.DataFrame, *, source: str, l_jun_nh: float | None = None
) -> pd.DataFrame:
    focus = filter_rows(frame, source=source, l_jun_nh=l_jun_nh)
    if focus.empty:
        return pd.DataFrame(
            columns=[
                "source",
                "l_jun_nh",
                "qubits",
                "trace_rows",
                "frequency_min_ghz",
                "frequency_max_ghz",
            ]
        )
    return pd.DataFrame(
        {
            "source": [source],
            "l_jun_nh": [l_jun_nh],
            "qubits": [",".join(sorted(focus["qubit"].astype(str).unique()))],
            "trace_rows": [len(focus)],
            "frequency_min_ghz": [pd.to_numeric(focus["frequency_ghz"], errors="coerce").min()],
            "frequency_max_ghz": [pd.to_numeric(focus["frequency_ghz"], errors="coerce").max()],
        }
    )


def c_eff_xy_tables(
    *,
    trend_df: pd.DataFrame,
    fit_df: pd.DataFrame,
    source: str,
    l_jun_nh: float,
    dataset: str,
    no_offset_fit_df: pd.DataFrame | None = None,
) -> list[dict[str, object]]:
    tables = build_c_eff_xy_t1_result_display_tables(
        trend_df=trend_df,
        fit_df=fit_df,
        no_offset_fit_df=no_offset_fit_df,
        source=source,
        l_jun_nh=float(l_jun_nh),
        dataset=dataset,
    )
    return [
        table_payload("Trend rows", tables["points"]),
        table_payload("Fit rows", tables["fits"]),
    ]


def register_result(
    result_key: str,
    *,
    title: str,
    purpose: str,
    figure,
    tables: list[dict[str, object]],
) -> None:
    if figure is None:
        raise ValueError(f"Result {result_key!r} has no figure object.")
    if not tables:
        raise ValueError(f"Result {result_key!r} has no table payload.")
    RESULTS[result_key] = {
        "title": title,
        "purpose": purpose,
        "figure_stem": result_key,
        "figure": figure,
        "tables": tables,
    }


def show_result(result_key: str) -> None:
    result = RESULTS[result_key]
    display(
        Markdown(
            f"#### {result['title']}\n\n"
            f"{result['purpose']}\n\n"
            f"Figure stem: `{result['figure_stem']}`"
        )
    )
    for table in result["tables"]:
        display(Markdown(f"**{table['title']}**"))
        display(table["data"])
    result["figure"].show(config=plotly_show_config(result["figure_stem"]))


focus_trace_fig = make_focus_admittance_trace_figure(
    circuit_trace_df=circuit_trace_df,
    focus_qubit=FOCUS_QUBIT,
    focus_l_jun_nh=FOCUS_LJUN_NH,
)
focus_resonance_fig = make_focus_resonance_extraction_figure(
    circuit_trace_df=circuit_trace_df,
    circuit_observables_df=circuit_observables_df,
    focus_qubit=FOCUS_QUBIT,
    focus_l_jun_nh=FOCUS_LJUN_NH,
)
im_trace_comparison_fig = make_im_trace_comparison_figure(
    layout_trace_df=layout_trace_df,
    circuit_trace_df=circuit_trace_df,
    trace_l_jun_nh_values=TRACE_L_JUN_NH_VALUES,
    max_points_per_line=PLOT_TRACE_MAX_POINTS_PER_LINE,
)
frequency_sweep_fig = make_resonance_frequency_sweep_figure(
    resonance_df=resonance_df,
    lc_fit_curve_df=lc_fit_curve_df,
)
frequency_ratio_fig = make_resonance_frequency_ratio_figure(resonance_df=resonance_df)
on_resonance_re_y_fig = make_on_resonance_re_y_figure(on_resonance_re_y_df)
t1_comparison_fig = make_t1_comparison_figure(t1_df)
c_eff_overview_fig = make_c_eff_overview_figure(lc_fit_params_df)
c_eff_reference_fig = make_c_eff_reference_diagnostic_figure(c_eff_reference_df)
re_y_ratio_fig = make_re_y_ratio_figure(on_resonance_re_y_df)

c_eff_xy_t1_trend_figures = {}
c_eff_xy_t1_trend_meta = {}
for source in C_EFF_XY_T1_SOURCES:
    source_key = source.lower()
    for l_jun_nh in C_EFF_XY_T1_L_JUN_NH_VALUES:
        has_successful_fit = (
            (c_eff_xy_t1_trend_fit_df["source"].astype(str) == source)
            & (c_eff_xy_t1_trend_fit_df["status"].astype(str) == "success")
            & (c_eff_xy_t1_trend_fit_df["l_jun_nh"].astype(float) - float(l_jun_nh)).abs().lt(1e-9)
        ).any()
        if not has_successful_fit:
            continue
        stem = f"{source_key}_c_eff_xy_t1_trend_{l_jun_figure_stem(l_jun_nh)}"
        no_offset_fit_df = c_eff_xy_t1_trend_fit_no_offset_df if source == "Circuit" else None
        c_eff_xy_t1_trend_figures[stem] = make_c_eff_xy_t1_trend_figure(
            trend_df=c_eff_xy_t1_trend_df,
            trend_fit_df=c_eff_xy_t1_trend_fit_df,
            no_offset_fit_df=no_offset_fit_df,
            l_jun_nh=float(l_jun_nh),
            source=source,
        )
        c_eff_xy_t1_trend_meta[stem] = {"source": source, "l_jun_nh": float(l_jun_nh)}

if not c_eff_xy_t1_trend_figures:
    raise ValueError(
        "No successful Ceff,xy T1 trend fits were available for the selected sources/L_jun values."
    )

c_eff_xy_t1_comparison_figures = {}
c_eff_xy_t1_comparison_meta = {}
for l_jun_nh in C_EFF_XY_T1_L_JUN_NH_VALUES:
    successful_sources = c_eff_xy_t1_trend_fit_df[
        (c_eff_xy_t1_trend_fit_df["status"].astype(str) == "success")
        & (c_eff_xy_t1_trend_fit_df["l_jun_nh"].astype(float) - float(l_jun_nh)).abs().lt(1e-9)
    ]["source"].astype(str)
    comparison_sources = [
        source for source in C_EFF_XY_T1_SOURCES if source in set(successful_sources)
    ]
    if len(comparison_sources) < 2:
        continue
    stem = f"layout_vs_circuit_c_eff_xy_t1_trend_{l_jun_figure_stem(l_jun_nh)}"
    c_eff_xy_t1_comparison_figures[stem] = make_c_eff_xy_t1_trend_comparison_figure(
        trend_df=c_eff_xy_t1_trend_df,
        trend_fit_df=c_eff_xy_t1_trend_fit_df,
        l_jun_nh=float(l_jun_nh),
        sources=comparison_sources,
    )
    c_eff_xy_t1_comparison_meta[stem] = {
        "l_jun_nh": float(l_jun_nh),
        "sources": comparison_sources,
    }

synthetic_c_eff_xy_t1_trend_figures = {}
synthetic_c_eff_xy_t1_trend_meta = {}
for l_jun_nh in SYNTHETIC_L_JUN_NH_VALUES:
    has_l_jun_rows = (
        (synthetic_c_eff_xy_t1_trend_df["source"].astype(str) == "Circuit")
        & (synthetic_c_eff_xy_t1_trend_df["l_jun_nh"].astype(float) - float(l_jun_nh))
        .abs()
        .lt(1e-9)
    ).any()
    if not has_l_jun_rows:
        continue
    stem = f"circuit_synthetic_c_eff_xy_t1_trend_{l_jun_figure_stem(l_jun_nh)}"
    synthetic_c_eff_xy_t1_trend_figures[stem] = make_c_eff_xy_t1_trend_figure(
        trend_df=synthetic_c_eff_xy_t1_trend_df,
        trend_fit_df=synthetic_c_eff_xy_t1_fit_df,
        no_offset_fit_df=synthetic_c_eff_xy_t1_fit_no_offset_df,
        l_jun_nh=float(l_jun_nh),
        title=f"Synthetic Circuit T1 vs Ceff,xy (Ljun={float(l_jun_nh):g} nH)",
        show_point_labels=False,
    )
    synthetic_c_eff_xy_t1_trend_meta[stem] = {"source": "Circuit", "l_jun_nh": float(l_jun_nh)}

if not synthetic_c_eff_xy_t1_trend_figures:
    raise ValueError("No synthetic Ceff,xy trend rows were available.")

synthetic_display_key = (
    f"circuit_synthetic_c_eff_xy_t1_trend_{l_jun_figure_stem(SYNTHETIC_DISPLAY_L_JUN_NH)}"
)
synthetic_display_fig = synthetic_c_eff_xy_t1_trend_figures.get(
    synthetic_display_key,
    next(iter(synthetic_c_eff_xy_t1_trend_figures.values())),
)
synthetic_c_eff_xy_t1_trace_names = [str(trace.name) for trace in synthetic_display_fig.data]
if not any(name.startswith("B fixed 0") for name in synthetic_c_eff_xy_t1_trace_names):
    raise RuntimeError(
        "Synthetic Ceff,xy/T1 figure is missing the B=0 fit. "
        "Restart the kernel, run the first import/config cell, then rerun this registry cell."
    )

RESULTS: dict[str, dict[str, object]] = {}

register_result(
    "q3d_jc_reduced_admittance_trace_Q0_L24nH",
    title="Circuit reduced admittance trace",
    purpose="Checks the reduced Re(Y)/Im(Y) trace at the focus qubit and Ljun before resonance extraction.",
    figure=focus_trace_fig,
    tables=[
        table_payload(
            "Focus trace summary",
            trace_summary(circuit_trace_df, source="Circuit", l_jun_nh=FOCUS_LJUN_NH),
        ),
        table_payload(
            "Focus observable row",
            filter_rows(
                circuit_observables_df, source="Circuit", qubit=FOCUS_QUBIT, l_jun_nh=FOCUS_LJUN_NH
            ),
            columns=[
                "source",
                "qubit",
                "l_jun_nh",
                "frequency_ghz",
                "re_y_s",
                "t1_us",
                "c_eff_q_ff",
                "fallback",
                "crossed",
            ],
        ),
    ],
)
register_result(
    "q3d_jc_resonance_extraction_Q0_L24nH",
    title="Circuit resonance extraction diagnostic",
    purpose="Shows the Im(Y) zero crossing and the extracted resonance for the same focus point.",
    figure=focus_resonance_fig,
    tables=[
        table_payload(
            "Extracted resonance row",
            filter_rows(
                circuit_observables_df, source="Circuit", qubit=FOCUS_QUBIT, l_jun_nh=FOCUS_LJUN_NH
            ),
            columns=[
                "source",
                "qubit",
                "l_jun_nh",
                "frequency_ghz",
                "im_y_s",
                "re_y_s",
                "fallback",
                "crossed",
            ],
        )
    ],
)
register_result(
    "layout_vs_circuit_im_y_trace_comparison",
    title="Im(Y) trace comparison",
    purpose="Compares Layout and Circuit Im(Y) traces at the configured trace Ljun values.",
    figure=im_trace_comparison_fig,
    tables=[
        table_payload(
            "Layout trace summary",
            trace_summary(layout_trace_df, source="Layout", l_jun_nh=FOCUS_LJUN_NH),
        ),
        table_payload(
            "Circuit trace summary",
            trace_summary(circuit_trace_df, source="Circuit", l_jun_nh=FOCUS_LJUN_NH),
        ),
        table_payload(
            "Frequency comparison rows",
            filter_rows(frequency_comparison_df, l_jun_nh=FOCUS_LJUN_NH),
            max_rows=12,
        ),
    ],
)
register_result(
    "layout_vs_circuit_resonance_frequency_lc_fit",
    title="LC-fit resonance frequency sweep",
    purpose="Shows resonance frequency versus Ljun with the LC-fit model overlay.",
    figure=frequency_sweep_fig,
    tables=[table_payload("LC frequency fit results", lc_frequency_fit_display_df)],
)
register_result(
    "layout_vs_circuit_resonance_frequency_ratio",
    title="Circuit/Layout resonance frequency ratio",
    purpose="Shows whether Circuit resonance frequencies track Layout frequencies across Ljun.",
    figure=frequency_ratio_fig,
    tables=[table_payload("Frequency ratio display table", frequency_ratio_display_df)],
)
register_result(
    "layout_vs_circuit_on_resonance_re_y",
    title="On-resonance Re(Y)",
    purpose="Compares the matched on-resonance Re(Y) values used for T1 calculation.",
    figure=on_resonance_re_y_fig,
    tables=[
        table_payload(
            "On-resonance Re(Y) rows",
            on_resonance_re_y_df,
            columns=[
                "source",
                "qubit",
                "l_jun_nh",
                "frequency_ghz",
                "re_y_frequency_ghz",
                "re_y_delta_mhz",
                "re_y_s",
                "re_y_matched",
                "t1_source",
            ],
        )
    ],
)
register_result(
    "layout_vs_circuit_t1_comparison",
    title="T1 comparison from LC-fit Ceff",
    purpose="Compares T1 values after applying each source's LC-fit effective capacitance.",
    figure=t1_comparison_fig,
    tables=[
        table_payload(
            "T1 rows",
            t1_df,
            columns=[
                "source",
                "qubit",
                "l_jun_nh",
                "frequency_ghz",
                "re_y_s",
                "C_eff_lc_fit_fF",
                "t1_from_lc_fit_us",
                "t1_from_q3d_ceff_us",
                "ratio_q3d_over_lc",
            ],
        )
    ],
)
register_result(
    "layout_vs_circuit_on_resonance_re_y_ratio",
    title="Circuit/Layout Re(Y) ratio",
    purpose="Shows the Circuit/Layout on-resonance Re(Y) ratio by qubit and Ljun.",
    figure=re_y_ratio_fig,
    tables=[table_payload("Re(Y) ratio display table", re_y_ratio_display_df)],
)
register_result(
    "layout_vs_circuit_c_eff_fit_overview",
    title="LC-fit effective capacitance overview",
    purpose="Summarizes fitted effective capacitance and source-aware Ls policy by source and qubit.",
    figure=c_eff_overview_fig,
    tables=[table_payload("LC frequency fit results", lc_frequency_fit_display_df)],
)
register_result(
    "layout_vs_circuit_c_eff_q3d_reference_diagnostic",
    title="LC-fit vs Q3D-reduced Ceff reference",
    purpose="Compares LC-fit Ceff against Q3D-reduced Ceff where that reference exists.",
    figure=c_eff_reference_fig,
    tables=[table_payload("Ceff reference comparison", c_eff_reference_df)],
)

for stem, figure in c_eff_xy_t1_trend_figures.items():
    meta = c_eff_xy_t1_trend_meta[stem]
    source = meta["source"]
    l_jun_nh = meta["l_jun_nh"]
    no_offset_fit_df = c_eff_xy_t1_trend_fit_no_offset_df if source == "Circuit" else None
    register_result(
        stem,
        title=f"{source} Ceff,xy/T1 trend at Ljun={l_jun_nh:g} nH",
        purpose="Displays the single-source inverse-square T1 trend and the exact points/fits used by this figure.",
        figure=figure,
        tables=c_eff_xy_tables(
            trend_df=c_eff_xy_t1_trend_df,
            fit_df=c_eff_xy_t1_trend_fit_df,
            no_offset_fit_df=no_offset_fit_df,
            source=source,
            l_jun_nh=l_jun_nh,
            dataset="Q3D",
        ),
    )

for stem, figure in c_eff_xy_t1_comparison_figures.items():
    meta = c_eff_xy_t1_comparison_meta[stem]
    l_jun_nh = meta["l_jun_nh"]
    tables = []
    for source in meta["sources"]:
        source_tables = c_eff_xy_tables(
            trend_df=c_eff_xy_t1_trend_df,
            fit_df=c_eff_xy_t1_trend_fit_df,
            source=source,
            l_jun_nh=l_jun_nh,
            dataset="Q3D",
        )
        for table in source_tables:
            table = dict(table)
            table["title"] = f"{source} {table['title']}"
            tables.append(table)
    register_result(
        stem,
        title=f"Layout vs Circuit Ceff,xy/T1 trend at Ljun={l_jun_nh:g} nH",
        purpose="Overlays the Layout and Circuit inverse-square T1 trends for the same Ljun.",
        figure=figure,
        tables=tables,
    )

for stem, figure in synthetic_c_eff_xy_t1_trend_figures.items():
    meta = synthetic_c_eff_xy_t1_trend_meta[stem]
    l_jun_nh = meta["l_jun_nh"]
    register_result(
        stem,
        title=f"Synthetic Circuit Ceff,xy/T1 trend at Ljun={l_jun_nh:g} nH",
        purpose="Displays the synthetic Circuit inverse-square T1 trend and both floating-B and B=0 fits.",
        figure=figure,
        tables=c_eff_xy_tables(
            trend_df=synthetic_c_eff_xy_t1_trend_df,
            fit_df=synthetic_c_eff_xy_t1_fit_df,
            no_offset_fit_df=synthetic_c_eff_xy_t1_fit_no_offset_df,
            source="Circuit",
            l_jun_nh=l_jun_nh,
            dataset="Synthetic Circuit",
        ),
    )

invalid_results = [
    key for key, result in RESULTS.items() if result["figure"] is None or not result["tables"]
]
if invalid_results:
    raise RuntimeError(f"Invalid result registry entries: {invalid_results}")

print(f"Registered {len(RESULTS)} result blocks.")


Registered 32 result blocks.


## Focus diagnostics


### Circuit reduced admittance trace

Purpose: Checks the raw reduced-admittance focus trace before resonance extraction.

Tables: Focus trace summary and the extracted observable row.

Result key: `q3d_jc_reduced_admittance_trace_Q0_L24nH`


In [4]:
show_result("q3d_jc_reduced_admittance_trace_Q0_L24nH")


#### Circuit reduced admittance trace

Checks the reduced Re(Y)/Im(Y) trace at the focus qubit and Ljun before resonance extraction.

Figure stem: `q3d_jc_reduced_admittance_trace_Q0_L24nH`

**Focus trace summary**

,source,l_jun_nh,qubits,trace_rows,frequency_min_ghz,frequency_max_ghz
0,Circuit,24.0,"Q0,Q1,Q2",543,1.0,10.0


**Focus observable row**

,source,qubit,l_jun_nh,frequency_ghz,re_y_s,t1_us,c_eff_q_ff,fallback,crossed
6,Circuit,Q0,24.0,4.392038,3.107467e-09,35.18632,109.340342,False,True


### Circuit resonance extraction diagnostic

Purpose: Shows the Im(Y) zero-crossing diagnostic for the focus point.

Tables: Extracted resonance and admittance row.

Result key: `q3d_jc_resonance_extraction_Q0_L24nH`


In [5]:
show_result("q3d_jc_resonance_extraction_Q0_L24nH")


#### Circuit resonance extraction diagnostic

Shows the Im(Y) zero crossing and the extracted resonance for the same focus point.

Figure stem: `q3d_jc_resonance_extraction_Q0_L24nH`

**Extracted resonance row**

,source,qubit,l_jun_nh,frequency_ghz,re_y_s,fallback,crossed
6,Circuit,Q0,24.0,4.392038,3.107467e-09,False,True


## Resonance frequency


### Im(Y) trace comparison

Purpose: Compares Layout and Circuit Im(Y) traces at the configured trace Ljun values.

Tables: Trace summaries and focus-frequency comparison rows.

Result key: `layout_vs_circuit_im_y_trace_comparison`


In [6]:
show_result("layout_vs_circuit_im_y_trace_comparison")


#### Im(Y) trace comparison

Compares Layout and Circuit Im(Y) traces at the configured trace Ljun values.

Figure stem: `layout_vs_circuit_im_y_trace_comparison`

**Layout trace summary**

,source,l_jun_nh,qubits,trace_rows,frequency_min_ghz,frequency_max_ghz
0,Layout,24.0,"Q0,Q1,Q2",75000,0.0,20.0


**Circuit trace summary**

,source,l_jun_nh,qubits,trace_rows,frequency_min_ghz,frequency_max_ghz
0,Circuit,24.0,"Q0,Q1,Q2",543,1.0,10.0


**Frequency comparison rows**

,qubit,l_jun_nh,Circuit,Layout,delta_circuit_minus_layout_mhz
6,Q0,24.0,4.392038,4.301763,90.274814
15,Q1,24.0,4.389998,4.287910,102.087442
24,Q2,24.0,4.394183,4.286585,107.598755


### LC-fit resonance frequency sweep

Purpose: Shows resonance frequency versus Ljun with the LC-fit model overlay.

Tables: LC frequency fit display table.

Result key: `layout_vs_circuit_resonance_frequency_lc_fit`


In [7]:
show_result("layout_vs_circuit_resonance_frequency_lc_fit")


#### LC-fit resonance frequency sweep

Shows resonance frequency versus Ljun with the LC-fit model overlay.

Figure stem: `layout_vs_circuit_resonance_frequency_lc_fit`

**LC frequency fit results**

,dataset,source,qubit,status,formula,L_jun_effective_factor,Ls_nH,Ls_policy,C_eff_lc_fit_pF,C_eff_lc_fit_fF,RMSE_GHz,RMSE_MHz,n_points,reason
0,Layout/Circuit,Circuit,Q0,success,f = 1 / (2*pi*sqrt((0.5*L_jun) * C_eff)),0.5,0.000000,fixed at 0 for reduced-circuit route,0.109428,109.427998,0.000028,0.028007,9,
1,Layout/Circuit,Circuit,Q1,success,f = 1 / (2*pi*sqrt((0.5*L_jun) * C_eff)),0.5,0.000000,fixed at 0 for reduced-circuit route,0.109530,109.530228,0.000030,0.029758,9,
2,Layout/Circuit,Circuit,Q2,success,f = 1 / (2*pi*sqrt((0.5*L_jun) * C_eff)),0.5,0.000000,fixed at 0 for reduced-circuit route,0.109321,109.320642,0.000027,0.027306,9,
3,Layout/Circuit,Layout,Q0,success,f = 1 / (2*pi*sqrt((Ls + 0.5*L_jun) * C_eff)),0.5,0.235131,fitted effective offset; not a separately iden...,0.111440,111.440260,0.004994,4.993587,9,
4,Layout/Circuit,Layout,Q1,success,f = 1 / (2*pi*sqrt((Ls + 0.5*L_jun) * C_eff)),0.5,0.229625,fitted effective offset; not a separately iden...,0.111911,111.911370,0.008203,8.202521,9,
5,Layout/Circuit,Layout,Q2,success,f = 1 / (2*pi*sqrt((Ls + 0.5*L_jun) * C_eff)),0.5,0.227581,fitted effective offset; not a separately iden...,0.111994,111.993936,0.008388,8.387555,9,


### Circuit/Layout resonance frequency ratio

Purpose: Shows whether Circuit resonance frequencies track Layout frequencies across Ljun.

Tables: Frequency ratio and MHz delta table.

Result key: `layout_vs_circuit_resonance_frequency_ratio`


In [8]:
show_result("layout_vs_circuit_resonance_frequency_ratio")


#### Circuit/Layout resonance frequency ratio

Shows whether Circuit resonance frequencies track Layout frequencies across Ljun.

Figure stem: `layout_vs_circuit_resonance_frequency_ratio`

**Frequency ratio display table**

,qubit,l_jun_nh,Circuit_frequency_GHz,Layout_frequency_GHz,ratio_circuit_over_layout,ratio_percent_offset,delta_circuit_minus_layout_mhz
0,Q0,5.0,9.622421,9.115459,1.055616,5.561568,506.962448
1,Q0,10.0,6.804070,6.591984,1.032173,3.217337,212.086339
2,Q0,15.0,5.555511,5.424259,1.024197,2.419716,131.251661
3,Q0,18.0,5.071505,4.951333,1.024271,2.427064,120.172021
4,Q0,20.0,4.811239,4.715740,1.020251,2.025119,95.499352
5,Q0,22.0,4.587346,4.501077,1.019166,1.916625,86.268769
6,Q0,24.0,4.392038,4.301763,1.020986,2.098554,90.274814
7,Q0,26.0,4.219767,4.147190,1.017500,1.750044,72.577638
8,Q0,28.0,4.066269,3.998906,1.016845,1.684521,67.362404
9,Q1,5.0,9.617933,9.104923,1.056344,5.634425,513.010011


## On-resonance loss and T1


### On-resonance Re(Y)

Purpose: Compares the on-resonance Re(Y) values used downstream for T1.

Tables: Matched Re(Y) rows.

Result key: `layout_vs_circuit_on_resonance_re_y`


In [9]:
show_result("layout_vs_circuit_on_resonance_re_y")


#### On-resonance Re(Y)

Compares the matched on-resonance Re(Y) values used for T1 calculation.

Figure stem: `layout_vs_circuit_on_resonance_re_y`

**On-resonance Re(Y) rows**

,source,qubit,l_jun_nh,frequency_ghz,re_y_frequency_ghz,re_y_delta_mhz,re_y_s,re_y_matched,t1_source
27,Circuit,Q0,5.0,9.622421,9.622421,0.000000,1.491543e-08,True,pending_lc_fit
28,Circuit,Q0,10.0,6.804070,6.804070,0.000000,7.457705e-09,True,pending_lc_fit
29,Circuit,Q0,15.0,5.555511,5.555511,0.000000,4.971849e-09,True,pending_lc_fit
30,Circuit,Q0,18.0,5.071505,5.071505,0.000000,4.143341e-09,True,pending_lc_fit
31,Circuit,Q0,20.0,4.811239,4.811239,0.000000,3.728970e-09,True,pending_lc_fit
32,Circuit,Q0,22.0,4.587346,4.587346,0.000000,3.389999e-09,True,pending_lc_fit
33,Circuit,Q0,24.0,4.392038,4.392038,0.000000,3.107467e-09,True,pending_lc_fit
34,Circuit,Q0,26.0,4.219767,4.219767,0.000000,2.868524e-09,True,pending_lc_fit
35,Circuit,Q0,28.0,4.066269,4.066269,0.000000,2.663628e-09,True,pending_lc_fit
36,Circuit,Q1,5.0,9.617933,9.617933,0.000000,7.999359e-09,True,pending_lc_fit


### T1 comparison from LC-fit Ceff

Purpose: Compares T1 values after applying each source's LC-fit effective capacitance.

Tables: T1 and Ceff rows.

Result key: `layout_vs_circuit_t1_comparison`


In [10]:
show_result("layout_vs_circuit_t1_comparison")


#### T1 comparison from LC-fit Ceff

Compares T1 values after applying each source's LC-fit effective capacitance.

Figure stem: `layout_vs_circuit_t1_comparison`

**T1 rows**

,source,qubit,l_jun_nh,frequency_ghz,re_y_s,C_eff_lc_fit_fF,t1_from_lc_fit_us,t1_from_q3d_ceff_us,ratio_q3d_over_lc
27,Circuit,Q0,5.0,9.622421,1.491543e-08,109.427998,7.336562,7.330685,0.999199
28,Circuit,Q0,10.0,6.804070,7.457705e-09,109.427998,14.673146,14.661392,0.999199
29,Circuit,Q0,15.0,5.555511,4.971849e-09,109.427998,22.009519,21.991889,0.999199
30,Circuit,Q0,18.0,5.071505,4.143341e-09,109.427998,26.410573,26.389417,0.999199
31,Circuit,Q0,20.0,4.811239,3.728970e-09,109.427998,29.345370,29.321864,0.999199
32,Circuit,Q0,22.0,4.587346,3.389999e-09,109.427998,32.279660,32.253802,0.999199
33,Circuit,Q0,24.0,4.392038,3.107467e-09,109.427998,35.214528,35.186320,0.999199
34,Circuit,Q0,26.0,4.219767,2.868524e-09,109.427998,38.147839,38.117281,0.999199
35,Circuit,Q0,28.0,4.066269,2.663628e-09,109.427998,41.082317,41.049408,0.999199
36,Circuit,Q1,5.0,9.617933,7.999359e-09,109.530228,13.692375,13.684704,0.999440


### Circuit/Layout Re(Y) ratio

Purpose: Shows the Circuit/Layout on-resonance Re(Y) ratio by qubit and Ljun.

Tables: Re(Y) ratio table.

Result key: `layout_vs_circuit_on_resonance_re_y_ratio`


In [11]:
show_result("layout_vs_circuit_on_resonance_re_y_ratio")


#### Circuit/Layout Re(Y) ratio

Shows the Circuit/Layout on-resonance Re(Y) ratio by qubit and Ljun.

Figure stem: `layout_vs_circuit_on_resonance_re_y_ratio`

**Re(Y) ratio display table**

,qubit,l_jun_nh,Circuit_re_y_s,Layout_re_y_s,ratio_circuit_over_layout
1,Q0,10.0,7.457705e-09,6.798281e-09,1.096999
2,Q0,15.0,4.971849e-09,4.561246e-09,1.090020
3,Q0,18.0,4.143341e-09,3.794121e-09,1.092042
4,Q0,20.0,3.728970e-09,3.440941e-09,1.083706
5,Q0,22.0,3.389999e-09,3.133671e-09,1.081798
6,Q0,24.0,3.107467e-09,2.862563e-09,1.085554
7,Q0,26.0,2.868524e-09,2.660151e-09,1.078331
8,Q0,28.0,2.663628e-09,2.473968e-09,1.076662
14,Q1,22.0,1.818110e-09,1.574820e-09,1.154488
15,Q1,24.0,1.666588e-09,1.432759e-09,1.163202


## Effective capacitance


### LC-fit effective capacitance overview

Purpose: Summarizes fitted Ceff and source-aware Ls policy by source and qubit.

Tables: LC frequency fit display table.

Result key: `layout_vs_circuit_c_eff_fit_overview`


In [12]:
show_result("layout_vs_circuit_c_eff_fit_overview")


#### LC-fit effective capacitance overview

Summarizes fitted effective capacitance and source-aware Ls policy by source and qubit.

Figure stem: `layout_vs_circuit_c_eff_fit_overview`

**LC frequency fit results**

,dataset,source,qubit,status,formula,L_jun_effective_factor,Ls_nH,Ls_policy,C_eff_lc_fit_pF,C_eff_lc_fit_fF,RMSE_GHz,RMSE_MHz,n_points,reason
0,Layout/Circuit,Circuit,Q0,success,f = 1 / (2*pi*sqrt((0.5*L_jun) * C_eff)),0.5,0.000000,fixed at 0 for reduced-circuit route,0.109428,109.427998,0.000028,0.028007,9,
1,Layout/Circuit,Circuit,Q1,success,f = 1 / (2*pi*sqrt((0.5*L_jun) * C_eff)),0.5,0.000000,fixed at 0 for reduced-circuit route,0.109530,109.530228,0.000030,0.029758,9,
2,Layout/Circuit,Circuit,Q2,success,f = 1 / (2*pi*sqrt((0.5*L_jun) * C_eff)),0.5,0.000000,fixed at 0 for reduced-circuit route,0.109321,109.320642,0.000027,0.027306,9,
3,Layout/Circuit,Layout,Q0,success,f = 1 / (2*pi*sqrt((Ls + 0.5*L_jun) * C_eff)),0.5,0.235131,fitted effective offset; not a separately iden...,0.111440,111.440260,0.004994,4.993587,9,
4,Layout/Circuit,Layout,Q1,success,f = 1 / (2*pi*sqrt((Ls + 0.5*L_jun) * C_eff)),0.5,0.229625,fitted effective offset; not a separately iden...,0.111911,111.911370,0.008203,8.202521,9,
5,Layout/Circuit,Layout,Q2,success,f = 1 / (2*pi*sqrt((Ls + 0.5*L_jun) * C_eff)),0.5,0.227581,fitted effective offset; not a separately iden...,0.111994,111.993936,0.008388,8.387555,9,


### LC-fit vs Q3D-reduced Ceff reference

Purpose: Compares LC-fit Ceff against the Q3D-reduced Ceff reference where available.

Tables: Ceff reference comparison table.

Result key: `layout_vs_circuit_c_eff_q3d_reference_diagnostic`


In [13]:
show_result("layout_vs_circuit_c_eff_q3d_reference_diagnostic")


#### LC-fit vs Q3D-reduced Ceff reference

Compares LC-fit Ceff against Q3D-reduced Ceff where that reference exists.

Figure stem: `layout_vs_circuit_c_eff_q3d_reference_diagnostic`

**Ceff reference comparison**

,source,qubit,status,Ls_nH,C_eff_lc_fit_fF,C_eff_q3d_reduction_fF,ratio_q3d_over_lc,RMSE_GHz,l_jun_effective_factor,n_points
0,Circuit,Q0,success,0.000000,109.427998,109.340342,0.999199,0.000028,0.5,9
1,Circuit,Q1,success,0.000000,109.530228,109.468862,0.999440,0.000030,0.5,9
2,Circuit,Q2,success,0.000000,109.320642,109.277035,0.999601,0.000027,0.5,9
3,Layout,Q0,success,0.235131,111.440260,NaN,NaN,0.004994,0.5,9
4,Layout,Q1,success,0.229625,111.911370,NaN,NaN,0.008203,0.5,9
5,Layout,Q2,success,0.227581,111.993936,NaN,NaN,0.008388,0.5,9


## Ceff,xy/T1 trend - Circuit per-Ljun


### Circuit Ceff,xy/T1 trend at Ljun=5 nH

Purpose: Shows one Circuit inverse-square T1 trend and the exact points/fits used by that figure.

Tables: Trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_c_eff_xy_t1_trend_L5nH`


In [14]:
show_result("circuit_c_eff_xy_t1_trend_L5nH")


#### Circuit Ceff,xy/T1 trend at Ljun=5 nH

Displays the single-source inverse-square T1 trend and the exact points/fits used by this figure.

Figure stem: `circuit_c_eff_xy_t1_trend_L5nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
18,Q3D,Circuit,Q2,5.0,9.627141,0.163967,0.163967,22.226008,109.320642
9,Q3D,Circuit,Q1,5.0,9.617933,0.209305,0.209305,13.692375,109.530228
0,Q3D,Circuit,Q0,5.0,9.622421,0.285673,0.285673,7.336562,109.427998


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Circuit,5.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,0.596797,0.040465,0.020686,0.999989,3,"Q0,Q1,Q2"
1,Q3D,Circuit,5.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),0.598220,0.000000,0.026033,0.999982,3,"Q0,Q1,Q2"


### Circuit Ceff,xy/T1 trend at Ljun=10 nH

Purpose: Shows one Circuit inverse-square T1 trend and the exact points/fits used by that figure.

Tables: Trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_c_eff_xy_t1_trend_L10nH`


In [15]:
show_result("circuit_c_eff_xy_t1_trend_L10nH")


#### Circuit Ceff,xy/T1 trend at Ljun=10 nH

Displays the single-source inverse-square T1 trend and the exact points/fits used by this figure.

Figure stem: `circuit_c_eff_xy_t1_trend_L10nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
19,Q3D,Circuit,Q2,10.0,6.807417,0.163967,0.163967,44.451934,109.320642
10,Q3D,Circuit,Q1,10.0,6.800887,0.209305,0.209305,27.384979,109.530228
1,Q3D,Circuit,Q0,10.0,6.804070,0.285673,0.285673,14.673146,109.427998


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Circuit,10.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,1.193589,0.08111,0.041489,0.999988,3,"Q0,Q1,Q2"
1,Q3D,Circuit,10.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),1.196442,0.00000,0.052201,0.999982,3,"Q0,Q1,Q2"


### Circuit Ceff,xy/T1 trend at Ljun=15 nH

Purpose: Shows one Circuit inverse-square T1 trend and the exact points/fits used by that figure.

Tables: Trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_c_eff_xy_t1_trend_L15nH`


In [16]:
show_result("circuit_c_eff_xy_t1_trend_L15nH")


#### Circuit Ceff,xy/T1 trend at Ljun=15 nH

Displays the single-source inverse-square T1 trend and the exact points/fits used by this figure.

Figure stem: `circuit_c_eff_xy_t1_trend_L15nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
20,Q3D,Circuit,Q2,15.0,5.558245,0.163967,0.163967,66.677291,109.320642
11,Q3D,Circuit,Q1,15.0,5.552911,0.209305,0.209305,41.077151,109.530228
2,Q3D,Circuit,Q0,15.0,5.555511,0.285673,0.285673,22.009519,109.427998


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Circuit,15.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,1.790366,0.121693,0.062260,0.999988,3,"Q0,Q1,Q2"
1,Q3D,Circuit,15.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),1.794647,0.000000,0.078329,0.999982,3,"Q0,Q1,Q2"


### Circuit Ceff,xy/T1 trend at Ljun=18 nH

Purpose: Shows one Circuit inverse-square T1 trend and the exact points/fits used by that figure.

Tables: Trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_c_eff_xy_t1_trend_L18nH`


In [17]:
show_result("circuit_c_eff_xy_t1_trend_L18nH")


#### Circuit Ceff,xy/T1 trend at Ljun=18 nH

Displays the single-source inverse-square T1 trend and the exact points/fits used by this figure.

Figure stem: `circuit_c_eff_xy_t1_trend_L18nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
21,Q3D,Circuit,Q2,18.0,5.073993,0.163967,0.163967,80.010635,109.320642
12,Q3D,Circuit,Q1,18.0,5.069138,0.209305,0.209305,49.290742,109.530228
3,Q3D,Circuit,Q0,18.0,5.071505,0.285673,0.285673,26.410573,109.427998


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Circuit,18.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,2.148391,0.145606,0.074500,0.999988,3,"Q0,Q1,Q2"
1,Q3D,Circuit,18.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),2.153513,0.000000,0.093726,0.999982,3,"Q0,Q1,Q2"


### Circuit Ceff,xy/T1 trend at Ljun=20 nH

Purpose: Shows one Circuit inverse-square T1 trend and the exact points/fits used by that figure.

Tables: Trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_c_eff_xy_t1_trend_L20nH`


In [18]:
show_result("circuit_c_eff_xy_t1_trend_L20nH")


#### Circuit Ceff,xy/T1 trend at Ljun=20 nH

Displays the single-source inverse-square T1 trend and the exact points/fits used by this figure.

Figure stem: `circuit_c_eff_xy_t1_trend_L20nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
22,Q3D,Circuit,Q2,20.0,4.813606,0.163967,0.163967,88.901215,109.320642
13,Q3D,Circuit,Q1,20.0,4.808989,0.209305,0.209305,54.768259,109.530228
4,Q3D,Circuit,Q0,20.0,4.811239,0.285673,0.285673,29.345370,109.427998


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Circuit,20.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,2.387108,0.162132,0.082958,0.999988,3,"Q0,Q1,Q2"
1,Q3D,Circuit,20.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),2.392811,0.000000,0.104365,0.999982,3,"Q0,Q1,Q2"


### Circuit Ceff,xy/T1 trend at Ljun=22 nH

Purpose: Shows one Circuit inverse-square T1 trend and the exact points/fits used by that figure.

Tables: Trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_c_eff_xy_t1_trend_L22nH`


In [19]:
show_result("circuit_c_eff_xy_t1_trend_L22nH")


#### Circuit Ceff,xy/T1 trend at Ljun=22 nH

Displays the single-source inverse-square T1 trend and the exact points/fits used by this figure.

Figure stem: `circuit_c_eff_xy_t1_trend_L22nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
23,Q3D,Circuit,Q2,22.0,4.589589,0.163967,0.163967,97.791660,109.320642
14,Q3D,Circuit,Q1,22.0,4.585212,0.209305,0.209305,60.244003,109.530228
5,Q3D,Circuit,Q0,22.0,4.587346,0.285673,0.285673,32.279660,109.427998


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Circuit,22.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,2.625846,0.177353,0.090748,0.999989,3,"Q0,Q1,Q2"
1,Q3D,Circuit,22.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),2.632085,0.000000,0.114165,0.999982,3,"Q0,Q1,Q2"


### Circuit Ceff,xy/T1 trend at Ljun=24 nH

Purpose: Shows one Circuit inverse-square T1 trend and the exact points/fits used by that figure.

Tables: Trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_c_eff_xy_t1_trend_L24nH`


In [20]:
show_result("circuit_c_eff_xy_t1_trend_L24nH")


#### Circuit Ceff,xy/T1 trend at Ljun=24 nH

Displays the single-source inverse-square T1 trend and the exact points/fits used by this figure.

Figure stem: `circuit_c_eff_xy_t1_trend_L24nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
24,Q3D,Circuit,Q2,24.0,4.394183,0.163967,0.163967,106.683140,109.320642
15,Q3D,Circuit,Q1,24.0,4.389998,0.209305,0.209305,65.721249,109.530228
6,Q3D,Circuit,Q0,24.0,4.392038,0.285673,0.285673,35.214528,109.427998


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Circuit,24.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,2.864600,0.193244,0.098882,0.999989,3,"Q0,Q1,Q2"
1,Q3D,Circuit,24.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),2.871397,0.000000,0.124396,0.999982,3,"Q0,Q1,Q2"


### Circuit Ceff,xy/T1 trend at Ljun=26 nH

Purpose: Shows one Circuit inverse-square T1 trend and the exact points/fits used by that figure.

Tables: Trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_c_eff_xy_t1_trend_L26nH`


In [21]:
show_result("circuit_c_eff_xy_t1_trend_L26nH")


#### Circuit Ceff,xy/T1 trend at Ljun=26 nH

Displays the single-source inverse-square T1 trend and the exact points/fits used by this figure.

Figure stem: `circuit_c_eff_xy_t1_trend_L26nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
25,Q3D,Circuit,Q2,26.0,4.221839,0.163967,0.163967,115.568516,109.320642
16,Q3D,Circuit,Q1,26.0,4.217797,0.209305,0.209305,71.196402,109.530228
7,Q3D,Circuit,Q0,26.0,4.219767,0.285673,0.285673,38.147839,109.427998


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Circuit,26.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,3.103165,0.21041,0.107668,0.999988,3,"Q0,Q1,Q2"
1,Q3D,Circuit,26.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),3.110567,0.00000,0.135448,0.999982,3,"Q0,Q1,Q2"


### Circuit Ceff,xy/T1 trend at Ljun=28 nH

Purpose: Shows one Circuit inverse-square T1 trend and the exact points/fits used by that figure.

Tables: Trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_c_eff_xy_t1_trend_L28nH`


In [22]:
show_result("circuit_c_eff_xy_t1_trend_L28nH")


#### Circuit Ceff,xy/T1 trend at Ljun=28 nH

Displays the single-source inverse-square T1 trend and the exact points/fits used by this figure.

Figure stem: `circuit_c_eff_xy_t1_trend_L28nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
26,Q3D,Circuit,Q2,28.0,4.068267,0.163967,0.163967,124.458275,109.320642
17,Q3D,Circuit,Q1,28.0,4.064368,0.209305,0.209305,76.673235,109.530228
8,Q3D,Circuit,Q0,28.0,4.066269,0.285673,0.285673,41.082317,109.427998


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Circuit,28.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,3.341863,0.226797,0.116055,0.999988,3,"Q0,Q1,Q2"
1,Q3D,Circuit,28.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),3.349841,0.000000,0.145998,0.999982,3,"Q0,Q1,Q2"


## Ceff,xy/T1 trend - Layout per-Ljun


### Layout Ceff,xy/T1 trend at Ljun=22 nH

Purpose: Shows one Layout inverse-square T1 trend and the exact points/fits used by that figure.

Tables: Trend rows plus Layout fit rows.

Result key: `layout_c_eff_xy_t1_trend_L22nH`


In [23]:
show_result("layout_c_eff_xy_t1_trend_L22nH")


#### Layout Ceff,xy/T1 trend at Ljun=22 nH

Displays the single-source inverse-square T1 trend and the exact points/fits used by this figure.

Figure stem: `layout_c_eff_xy_t1_trend_L22nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
37,Q3D,Layout,Q2,22.0,4.493618,0.163967,0.163967,109.442659,111.993936
35,Q3D,Layout,Q1,22.0,4.494705,0.209305,0.209305,71.062978,111.911370
31,Q3D,Layout,Q0,22.0,4.501077,0.285673,0.285673,35.562206,111.440260


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Layout,22.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,2.945268,1.066122,1.963951,0.995762,3,"Q0,Q1,Q2"


### Layout Ceff,xy/T1 trend at Ljun=24 nH

Purpose: Shows one Layout inverse-square T1 trend and the exact points/fits used by that figure.

Tables: Trend rows plus Layout fit rows.

Result key: `layout_c_eff_xy_t1_trend_L24nH`


In [24]:
show_result("layout_c_eff_xy_t1_trend_L24nH")


#### Layout Ceff,xy/T1 trend at Ljun=24 nH

Displays the single-source inverse-square T1 trend and the exact points/fits used by this figure.

Figure stem: `layout_c_eff_xy_t1_trend_L24nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
38,Q3D,Layout,Q2,24.0,4.286585,0.163967,0.163967,120.382390,111.993936
36,Q3D,Layout,Q1,24.0,4.287910,0.209305,0.209305,78.108988,111.911370
32,Q3D,Layout,Q0,24.0,4.301763,0.285673,0.285673,38.930236,111.440260


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Layout,24.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,3.246956,0.915865,2.183849,0.995689,3,"Q0,Q1,Q2"


## Ceff,xy/T1 trend - Layout-vs-Circuit comparison


### Layout vs Circuit Ceff,xy/T1 trend at Ljun=22 nH

Purpose: Overlays Layout and Circuit inverse-square T1 trends for the same Ljun.

Tables: Per-source trend rows and fit rows.

Result key: `layout_vs_circuit_c_eff_xy_t1_trend_L22nH`


In [25]:
show_result("layout_vs_circuit_c_eff_xy_t1_trend_L22nH")


#### Layout vs Circuit Ceff,xy/T1 trend at Ljun=22 nH

Overlays the Layout and Circuit inverse-square T1 trends for the same Ljun.

Figure stem: `layout_vs_circuit_c_eff_xy_t1_trend_L22nH`

**Circuit Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
23,Q3D,Circuit,Q2,22.0,4.589589,0.163967,0.163967,97.791660,109.320642
14,Q3D,Circuit,Q1,22.0,4.585212,0.209305,0.209305,60.244003,109.530228
5,Q3D,Circuit,Q0,22.0,4.587346,0.285673,0.285673,32.279660,109.427998


**Circuit Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Circuit,22.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,2.625846,0.177353,0.090748,0.999989,3,"Q0,Q1,Q2"


**Layout Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
37,Q3D,Layout,Q2,22.0,4.493618,0.163967,0.163967,109.442659,111.993936
35,Q3D,Layout,Q1,22.0,4.494705,0.209305,0.209305,71.062978,111.911370
31,Q3D,Layout,Q0,22.0,4.501077,0.285673,0.285673,35.562206,111.440260


**Layout Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Layout,22.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,2.945268,1.066122,1.963951,0.995762,3,"Q0,Q1,Q2"


### Layout vs Circuit Ceff,xy/T1 trend at Ljun=24 nH

Purpose: Overlays Layout and Circuit inverse-square T1 trends for the same Ljun.

Tables: Per-source trend rows and fit rows.

Result key: `layout_vs_circuit_c_eff_xy_t1_trend_L24nH`


In [26]:
show_result("layout_vs_circuit_c_eff_xy_t1_trend_L24nH")


#### Layout vs Circuit Ceff,xy/T1 trend at Ljun=24 nH

Overlays the Layout and Circuit inverse-square T1 trends for the same Ljun.

Figure stem: `layout_vs_circuit_c_eff_xy_t1_trend_L24nH`

**Circuit Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
24,Q3D,Circuit,Q2,24.0,4.394183,0.163967,0.163967,106.683140,109.320642
15,Q3D,Circuit,Q1,24.0,4.389998,0.209305,0.209305,65.721249,109.530228
6,Q3D,Circuit,Q0,24.0,4.392038,0.285673,0.285673,35.214528,109.427998


**Circuit Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Circuit,24.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,2.8646,0.193244,0.098882,0.999989,3,"Q0,Q1,Q2"


**Layout Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
38,Q3D,Layout,Q2,24.0,4.286585,0.163967,0.163967,120.382390,111.993936
36,Q3D,Layout,Q1,24.0,4.287910,0.209305,0.209305,78.108988,111.911370
32,Q3D,Layout,Q0,24.0,4.301763,0.285673,0.285673,38.930236,111.440260


**Layout Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Q3D,Layout,24.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,3.246956,0.915865,2.183849,0.995689,3,"Q0,Q1,Q2"


## Ceff,xy/T1 trend - Synthetic Circuit


### Synthetic Circuit Ceff,xy/T1 trend at Ljun=5 nH

Purpose: Shows one synthetic inverse-square T1 trend and both floating-B and B=0 fits.

Tables: Synthetic trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_synthetic_c_eff_xy_t1_trend_L5nH`


In [27]:
show_result("circuit_synthetic_c_eff_xy_t1_trend_L5nH")


#### Synthetic Circuit Ceff,xy/T1 trend at Ljun=5 nH

Displays the synthetic Circuit inverse-square T1 trend and both floating-B and B=0 fits.

Figure stem: `circuit_synthetic_c_eff_xy_t1_trend_L5nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
0,Synthetic Circuit,Circuit,S00,5.0,9.622432,0.10,0.10,59.872640,109.427743
9,Synthetic Circuit,Circuit,S01,5.0,9.622429,0.14,0.14,30.547312,109.427827
18,Synthetic Circuit,Circuit,S02,5.0,9.622426,0.18,0.18,18.479261,109.427895
27,Synthetic Circuit,Circuit,S03,5.0,9.622423,0.22,0.22,12.370426,109.427947
36,Synthetic Circuit,Circuit,S04,5.0,9.622422,0.26,0.26,8.856938,109.427983
45,Synthetic Circuit,Circuit,S05,5.0,9.622421,0.30,0.30,6.652547,109.428004
54,Synthetic Circuit,Circuit,S06,5.0,9.622421,0.34,0.34,5.179319,109.428009
63,Synthetic Circuit,Circuit,S07,5.0,9.622421,0.38,0.38,4.146324,109.427998
72,Synthetic Circuit,Circuit,S08,5.0,9.622422,0.42,0.42,3.394155,109.427971


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Synthetic Circuit,Circuit,5.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,0.598726,0.000035,0.000015,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"
1,Synthetic Circuit,Circuit,5.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),0.598727,0.000000,0.000030,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"


### Synthetic Circuit Ceff,xy/T1 trend at Ljun=10 nH

Purpose: Shows one synthetic inverse-square T1 trend and both floating-B and B=0 fits.

Tables: Synthetic trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_synthetic_c_eff_xy_t1_trend_L10nH`


In [28]:
show_result("circuit_synthetic_c_eff_xy_t1_trend_L10nH")


#### Synthetic Circuit Ceff,xy/T1 trend at Ljun=10 nH

Displays the synthetic Circuit inverse-square T1 trend and both floating-B and B=0 fits.

Figure stem: `circuit_synthetic_c_eff_xy_t1_trend_L10nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
1,Synthetic Circuit,Circuit,S00,10.0,6.804078,0.10,0.10,119.745455,109.427743
10,Synthetic Circuit,Circuit,S01,10.0,6.804075,0.14,0.14,61.094713,109.427827
19,Synthetic Circuit,Circuit,S02,10.0,6.804073,0.18,0.18,36.958576,109.427895
28,Synthetic Circuit,Circuit,S03,10.0,6.804072,0.22,0.22,24.740889,109.427947
37,Synthetic Circuit,Circuit,S04,10.0,6.804071,0.26,0.26,17.713903,109.427983
46,Synthetic Circuit,Circuit,S05,10.0,6.804070,0.30,0.30,13.305114,109.428004
55,Synthetic Circuit,Circuit,S06,10.0,6.804070,0.34,0.34,10.358654,109.428009
64,Synthetic Circuit,Circuit,S07,10.0,6.804070,0.38,0.38,8.292660,109.427998
73,Synthetic Circuit,Circuit,S08,10.0,6.804071,0.42,0.42,6.788319,109.427971


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Synthetic Circuit,Circuit,10.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,1.197454,0.00007,0.000031,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"
1,Synthetic Circuit,Circuit,10.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),1.197455,0.00000,0.000059,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"


### Synthetic Circuit Ceff,xy/T1 trend at Ljun=15 nH

Purpose: Shows one synthetic inverse-square T1 trend and both floating-B and B=0 fits.

Tables: Synthetic trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_synthetic_c_eff_xy_t1_trend_L15nH`


In [29]:
show_result("circuit_synthetic_c_eff_xy_t1_trend_L15nH")


#### Synthetic Circuit Ceff,xy/T1 trend at Ljun=15 nH

Displays the synthetic Circuit inverse-square T1 trend and both floating-B and B=0 fits.

Figure stem: `circuit_synthetic_c_eff_xy_t1_trend_L15nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
2,Synthetic Circuit,Circuit,S00,15.0,5.555517,0.10,0.10,179.616554,109.427743
11,Synthetic Circuit,Circuit,S01,15.0,5.555515,0.14,0.14,91.641240,109.427827
20,Synthetic Circuit,Circuit,S02,15.0,5.555513,0.18,0.18,55.437362,109.427895
29,Synthetic Circuit,Circuit,S03,15.0,5.555512,0.22,0.22,37.110997,109.427947
38,Synthetic Circuit,Circuit,S04,15.0,5.555511,0.26,0.26,26.570613,109.427983
47,Synthetic Circuit,Circuit,S05,15.0,5.555511,0.30,0.30,19.957490,109.428004
56,Synthetic Circuit,Circuit,S06,15.0,5.555510,0.34,0.34,15.537840,109.428009
65,Synthetic Circuit,Circuit,S07,15.0,5.555511,0.38,0.38,12.438878,109.427998
74,Synthetic Circuit,Circuit,S08,15.0,5.555511,0.42,0.42,10.182387,109.427971


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Synthetic Circuit,Circuit,15.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,1.796165,0.000106,0.000046,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"
1,Synthetic Circuit,Circuit,15.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),1.796167,0.000000,0.000089,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"


### Synthetic Circuit Ceff,xy/T1 trend at Ljun=18 nH

Purpose: Shows one synthetic inverse-square T1 trend and both floating-B and B=0 fits.

Tables: Synthetic trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_synthetic_c_eff_xy_t1_trend_L18nH`


In [30]:
show_result("circuit_synthetic_c_eff_xy_t1_trend_L18nH")


#### Synthetic Circuit Ceff,xy/T1 trend at Ljun=18 nH

Displays the synthetic Circuit inverse-square T1 trend and both floating-B and B=0 fits.

Figure stem: `circuit_synthetic_c_eff_xy_t1_trend_L18nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
3,Synthetic Circuit,Circuit,S00,18.0,5.071510,0.10,0.10,215.532925,109.427743
12,Synthetic Circuit,Circuit,S01,18.0,5.071509,0.14,0.14,109.965947,109.427827
21,Synthetic Circuit,Circuit,S02,18.0,5.071507,0.18,0.18,66.522692,109.427895
30,Synthetic Circuit,Circuit,S03,18.0,5.071506,0.22,0.22,44.531762,109.427947
39,Synthetic Circuit,Circuit,S04,18.0,5.071505,0.26,0.26,31.883709,109.427983
48,Synthetic Circuit,Circuit,S05,18.0,5.071504,0.30,0.30,23.948217,109.428004
57,Synthetic Circuit,Circuit,S06,18.0,5.071504,0.34,0.34,18.644807,109.428009
66,Synthetic Circuit,Circuit,S07,18.0,5.071505,0.38,0.38,14.926172,109.427998
75,Synthetic Circuit,Circuit,S08,18.0,5.071505,0.42,0.42,12.218471,109.427971


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Synthetic Circuit,Circuit,18.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,2.155329,0.000127,0.000055,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"
1,Synthetic Circuit,Circuit,18.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),2.155331,0.000000,0.000107,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"


### Synthetic Circuit Ceff,xy/T1 trend at Ljun=20 nH

Purpose: Shows one synthetic inverse-square T1 trend and both floating-B and B=0 fits.

Tables: Synthetic trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_synthetic_c_eff_xy_t1_trend_L20nH`


In [31]:
show_result("circuit_synthetic_c_eff_xy_t1_trend_L20nH")


#### Synthetic Circuit Ceff,xy/T1 trend at Ljun=20 nH

Displays the synthetic Circuit inverse-square T1 trend and both floating-B and B=0 fits.

Figure stem: `circuit_synthetic_c_eff_xy_t1_trend_L20nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
4,Synthetic Circuit,Circuit,S00,20.0,4.811245,0.10,0.10,239.483390,109.427743
13,Synthetic Circuit,Circuit,S01,20.0,4.811243,0.14,0.14,122.185591,109.427827
22,Synthetic Circuit,Circuit,S02,20.0,4.811242,0.18,0.18,73.914832,109.427895
31,Synthetic Circuit,Circuit,S03,20.0,4.811241,0.22,0.22,49.480224,109.427947
40,Synthetic Circuit,Circuit,S04,20.0,4.811240,0.26,0.26,35.426693,109.427983
49,Synthetic Circuit,Circuit,S05,20.0,4.811239,0.30,0.30,26.609393,109.428004
58,Synthetic Circuit,Circuit,S06,20.0,4.811239,0.34,0.34,20.716657,109.428009
67,Synthetic Circuit,Circuit,S07,20.0,4.811239,0.38,0.38,16.584800,109.427998
76,Synthetic Circuit,Circuit,S08,20.0,4.811240,0.42,0.42,13.576213,109.427971


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Synthetic Circuit,Circuit,20.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,2.394833,0.000141,0.000062,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"
1,Synthetic Circuit,Circuit,20.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),2.394836,0.000000,0.000119,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"


### Synthetic Circuit Ceff,xy/T1 trend at Ljun=22 nH

Purpose: Shows one synthetic inverse-square T1 trend and both floating-B and B=0 fits.

Tables: Synthetic trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_synthetic_c_eff_xy_t1_trend_L22nH`


In [32]:
show_result("circuit_synthetic_c_eff_xy_t1_trend_L22nH")


#### Synthetic Circuit Ceff,xy/T1 trend at Ljun=22 nH

Displays the synthetic Circuit inverse-square T1 trend and both floating-B and B=0 fits.

Figure stem: `circuit_synthetic_c_eff_xy_t1_trend_L22nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
5,Synthetic Circuit,Circuit,S00,22.0,4.587351,0.10,0.10,263.429715,109.427743
14,Synthetic Circuit,Circuit,S01,22.0,4.587349,0.14,0.14,134.403120,109.427827
23,Synthetic Circuit,Circuit,S02,22.0,4.587348,0.18,0.18,81.305692,109.427895
32,Synthetic Circuit,Circuit,S03,22.0,4.587347,0.22,0.22,54.427829,109.427947
41,Synthetic Circuit,Circuit,S04,22.0,4.587346,0.26,0.26,38.969063,109.427983
50,Synthetic Circuit,Circuit,S05,22.0,4.587346,0.30,0.30,29.270107,109.428004
59,Synthetic Circuit,Circuit,S06,22.0,4.587345,0.34,0.34,22.788148,109.428009
68,Synthetic Circuit,Circuit,S07,22.0,4.587346,0.38,0.38,18.243140,109.427998
77,Synthetic Circuit,Circuit,S08,22.0,4.587346,0.42,0.42,14.933719,109.427971


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Synthetic Circuit,Circuit,22.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,2.634296,0.000154,0.000067,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"
1,Synthetic Circuit,Circuit,22.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),2.634299,0.000000,0.000130,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"


### Synthetic Circuit Ceff,xy/T1 trend at Ljun=24 nH

Purpose: Shows one synthetic inverse-square T1 trend and both floating-B and B=0 fits.

Tables: Synthetic trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_synthetic_c_eff_xy_t1_trend_L24nH`


In [33]:
show_result("circuit_synthetic_c_eff_xy_t1_trend_L24nH")


#### Synthetic Circuit Ceff,xy/T1 trend at Ljun=24 nH

Displays the synthetic Circuit inverse-square T1 trend and both floating-B and B=0 fits.

Figure stem: `circuit_synthetic_c_eff_xy_t1_trend_L24nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
6,Synthetic Circuit,Circuit,S00,24.0,4.392043,0.10,0.10,287.380762,109.427743
15,Synthetic Circuit,Circuit,S01,24.0,4.392041,0.14,0.14,146.623060,109.427827
24,Synthetic Circuit,Circuit,S02,24.0,4.392040,0.18,0.18,88.698010,109.427895
33,Synthetic Circuit,Circuit,S03,24.0,4.392039,0.22,0.22,59.376410,109.427947
42,Synthetic Circuit,Circuit,S04,24.0,4.392038,0.26,0.26,42.512133,109.427983
51,Synthetic Circuit,Circuit,S05,24.0,4.392038,0.30,0.30,31.931347,109.428004
60,Synthetic Circuit,Circuit,S06,24.0,4.392038,0.34,0.34,24.860048,109.428009
69,Synthetic Circuit,Circuit,S07,24.0,4.392038,0.38,0.38,19.901807,109.427998
78,Synthetic Circuit,Circuit,S08,24.0,4.392038,0.42,0.42,16.291494,109.427971


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Synthetic Circuit,Circuit,24.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,2.873807,0.000168,0.000073,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"
1,Synthetic Circuit,Circuit,24.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),2.873810,0.000000,0.000142,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"


### Synthetic Circuit Ceff,xy/T1 trend at Ljun=26 nH

Purpose: Shows one synthetic inverse-square T1 trend and both floating-B and B=0 fits.

Tables: Synthetic trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_synthetic_c_eff_xy_t1_trend_L26nH`


In [34]:
show_result("circuit_synthetic_c_eff_xy_t1_trend_L26nH")


#### Synthetic Circuit Ceff,xy/T1 trend at Ljun=26 nH

Displays the synthetic Circuit inverse-square T1 trend and both floating-B and B=0 fits.

Figure stem: `circuit_synthetic_c_eff_xy_t1_trend_L26nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
7,Synthetic Circuit,Circuit,S00,26.0,4.219772,0.10,0.10,311.319086,109.427743
16,Synthetic Circuit,Circuit,S01,26.0,4.219771,0.14,0.14,158.836512,109.427827
25,Synthetic Circuit,Circuit,S02,26.0,4.219769,0.18,0.18,96.086404,109.427895
34,Synthetic Circuit,Circuit,S03,26.0,4.219768,0.22,0.22,64.322365,109.427947
43,Synthetic Circuit,Circuit,S04,26.0,4.219768,0.26,0.26,46.053322,109.427983
52,Synthetic Circuit,Circuit,S05,26.0,4.219767,0.30,0.30,34.591175,109.428004
61,Synthetic Circuit,Circuit,S06,26.0,4.219767,0.34,0.34,26.930848,109.428009
70,Synthetic Circuit,Circuit,S07,26.0,4.219767,0.38,0.38,21.559594,109.427998
79,Synthetic Circuit,Circuit,S08,26.0,4.219768,0.42,0.42,17.648548,109.427971


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Synthetic Circuit,Circuit,26.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,3.113190,0.000183,0.000080,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"
1,Synthetic Circuit,Circuit,26.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),3.113193,0.000000,0.000154,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"


### Synthetic Circuit Ceff,xy/T1 trend at Ljun=28 nH

Purpose: Shows one synthetic inverse-square T1 trend and both floating-B and B=0 fits.

Tables: Synthetic trend rows plus floating-B and B=0 fit rows.

Result key: `circuit_synthetic_c_eff_xy_t1_trend_L28nH`


In [35]:
show_result("circuit_synthetic_c_eff_xy_t1_trend_L28nH")


#### Synthetic Circuit Ceff,xy/T1 trend at Ljun=28 nH

Displays the synthetic Circuit inverse-square T1 trend and both floating-B and B=0 fits.

Figure stem: `circuit_synthetic_c_eff_xy_t1_trend_L28nH`

**Trend rows**

,dataset,source,qubit,l_jun_nh,frequency_ghz,C_eff_xy_signed_fF,C_eff_xy_abs_fF,t1_from_lc_fit_us,C_eff_lc_fit_fF
8,Synthetic Circuit,Circuit,S00,28.0,4.066273,0.10,0.10,335.266943,109.427743
17,Synthetic Circuit,Circuit,S01,28.0,4.066272,0.14,0.14,171.054824,109.427827
26,Synthetic Circuit,Circuit,S02,28.0,4.066271,0.18,0.18,103.477738,109.427895
35,Synthetic Circuit,Circuit,S03,28.0,4.066270,0.22,0.22,69.270288,109.427947
44,Synthetic Circuit,Circuit,S04,28.0,4.066269,0.26,0.26,49.595919,109.427983
53,Synthetic Circuit,Circuit,S05,28.0,4.066269,0.30,0.30,37.252060,109.428004
62,Synthetic Circuit,Circuit,S06,28.0,4.066268,0.34,0.34,29.002472,109.428009
71,Synthetic Circuit,Circuit,S07,28.0,4.066269,0.38,0.38,23.218041,109.427998
80,Synthetic Circuit,Circuit,S08,28.0,4.066269,0.42,0.42,19.006142,109.427971


**Fit rows**

,dataset,source,l_jun_nh,intercept_policy,formula,A_us_fF2,B_us,RMSE_us,R2,n_points,qubits
0,Synthetic Circuit,Circuit,28.0,free,T1_us = A_us_fF2 / Ceff_xy_fF^2 + B_us,3.352668,0.000197,0.000086,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"
1,Synthetic Circuit,Circuit,28.0,zero,T1_us = A_us_fF2 / Ceff_xy_fF^2 (B_us fixed to 0),3.352672,0.000000,0.000166,1.0,9,"S00,S01,S02,S03,S04,S05,S06,S07,S08"


## Saved Interactive Artifacts

Writes the same figure objects from the result registry to the existing Plotly artifact directory.


In [36]:
figures = {
    "q3d_jc_reduced_admittance_trace_Q0_L24nH": focus_trace_fig,
    "q3d_jc_resonance_extraction_Q0_L24nH": focus_resonance_fig,
    "layout_vs_circuit_im_y_trace_comparison": im_trace_comparison_fig,
    "layout_vs_circuit_resonance_frequency_lc_fit": frequency_sweep_fig,
    "layout_vs_circuit_resonance_frequency_ratio": frequency_ratio_fig,
    "layout_vs_circuit_on_resonance_re_y": on_resonance_re_y_fig,
    "layout_vs_circuit_t1_comparison": t1_comparison_fig,
    "layout_vs_circuit_c_eff_fit_overview": c_eff_overview_fig,
    "layout_vs_circuit_c_eff_q3d_reference_diagnostic": c_eff_reference_fig,
    "layout_vs_circuit_on_resonance_re_y_ratio": re_y_ratio_fig,
    **c_eff_xy_t1_trend_figures,
    **c_eff_xy_t1_comparison_figures,
    **synthetic_c_eff_xy_t1_trend_figures,
}
missing_result_keys = sorted(set(figures) - set(RESULTS))
if missing_result_keys:
    raise RuntimeError(f"Figures missing from result registry: {missing_result_keys}")
written_artifacts = write_plotly_artifacts(
    figures,
    figure_dir=FIG_DIR,
    write_html=WRITE_INTERACTIVE_HTML,
)
for artifact_path in written_artifacts:
    print(artifact_path.relative_to(REPO_ROOT))


sandbox/thesis_pf6fq_external_coupling_analysis/outputs/figures/q3d_jc_reduced_admittance_trace_Q0_L24nH.html
sandbox/thesis_pf6fq_external_coupling_analysis/outputs/figures/q3d_jc_resonance_extraction_Q0_L24nH.html
sandbox/thesis_pf6fq_external_coupling_analysis/outputs/figures/layout_vs_circuit_im_y_trace_comparison.html
sandbox/thesis_pf6fq_external_coupling_analysis/outputs/figures/layout_vs_circuit_resonance_frequency_lc_fit.html
sandbox/thesis_pf6fq_external_coupling_analysis/outputs/figures/layout_vs_circuit_resonance_frequency_ratio.html
sandbox/thesis_pf6fq_external_coupling_analysis/outputs/figures/layout_vs_circuit_on_resonance_re_y.html
sandbox/thesis_pf6fq_external_coupling_analysis/outputs/figures/layout_vs_circuit_t1_comparison.html
sandbox/thesis_pf6fq_external_coupling_analysis/outputs/figures/layout_vs_circuit_c_eff_fit_overview.html
sandbox/thesis_pf6fq_external_coupling_analysis/outputs/figures/layout_vs_circuit_c_eff_q3d_reference_diagnostic.html
sandbox/thesis_pf6